- ShortConv
    - 「短」修饰的是时间感受野（temporal receptive field）
    -  K3 的实际尺度上看这个对比很悬殊：short_conv_kernel_size = 4，而 max_position_embeddings = 1048576——核只覆盖 $4/2^{20}\approx 4\times10^{-6}$ 的上下文。
    -  nn.conv1d
        -  逐通道卷积（depthwise convolution）：
        -  PyTorch 做的是互相关（cross-correlation），不会把 kernel 反转。
            - np.correlate，而不是 np.convolve
```
输入 channel 0 ── theta[0] ──> 输出 channel 0
输入 channel 1 ── theta[1] ──> 输出 channel 1
输入 channel 2 ── theta[2] ──> 输出 channel 2
输入 channel 3 ── theta[3] ──> 输出 channel 3
```
- 把 KDA 的 q/k/v 参数化写成
    - $\boldsymbol q^h_t,\boldsymbol k^h_t=\mathrm{L_2Norm}(\mathrm{Swish}(\mathrm{ShortConv}(\mathbf W^h_{q/k}\boldsymbol x_t)))$
    - $\boldsymbol v^h_t=\mathrm{Swish}(\mathrm{ShortConv}(\mathbf W^h_v\boldsymbol x_t))$
- `F.silu`：Sigmoid Linear Unit (SiLU) function, element-wise.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

W, T, D, B = 4, 6, 4, 1

In [3]:
# U = q_proj(x), the tensor that actually enters q_conv1d: [B, T, D]
U = torch.tensor([[[1., 2., 1., 1.],
                   [-1., 2., 3., 2.],
                   [2., 3., 1., -2.],
                   [1., 1., 3., 1.],
                   [1., -1., 1., 2.],
                   [3., 2., 3., -1.]]])

In [4]:
U.shape

torch.Size([1, 6, 4])

In [5]:
# theta[d, j], j = 0..W-1 aligned to source positions t-W+1 .. t
theta = torch.tensor([[0.25, -0.50, 0.50, 1.00],
                      [-0.25, 0.50, 0.25, 0.50],
                      [0.50, 0.25, -0.50, 1.00],
                      [-0.50, 0.25, 0.50, 0.75]])
theta.shape

torch.Size([4, 4])

### 1d

In [25]:
# (B, C, L)
x = torch.tensor([1., 3., 1., 3., 1., 3.]).view(1, 1, -1)
k = torch.tensor([0.5, 0.25, -0.5, 1.0])
conv = nn.Conv1d(1, 1, kernel_size=4, padding=3, bias=False)
with torch.no_grad():
    conv.weight.copy_(k.view(1, 1, 4))
full_output = conv(x)                   # [1, 1, 9]
causal_output = full_output[..., :6]   # [1, 1, 6]
causal_output

tensor([[[ 1.0000,  2.5000, -0.2500,  3.7500,  1.2500,  3.7500]]],
       grad_fn=<SliceBackward0>)

- 前六个位置 $t=0,\ldots,5$ 是因果输出（causal output）。最后三个位置使用了右侧补零，对应图中的灰色“越界”结果，因此用：
    - `causal_output = full_output[..., :T]`

In [26]:
# channel 2
np.correlate([1, 3, 1, 3, 1, 3], [0.5, 0.25, -0.5, 1], mode='full')

array([ 1.  ,  2.5 , -0.25,  3.75,  1.25,  3.75,  0.25,  1.25,  1.5 ])

In [30]:
# channel 0
np.correlate([1, -1, 2, 1, 1, 3], [0.25, -0.5, 0.5, 1], mode='full')

array([ 1.  , -0.5 ,  1.  ,  2.75,  0.25,  3.5 ,  1.25, -1.25,  0.75])

In [29]:
# channel 1
np.correlate([2, 2, 3, 1, -1, 2], [-0.5, 0.5, 0.25, 0.5], mode='full')

array([ 1.  ,  1.5 ,  3.  ,  1.25,  0.25, -0.25, -0.5 ,  1.5 , -1.  ])

In [31]:
# channel 3

In [32]:
np.correlate([1, 2, -2, 1, 2, -1], [-0.50, 0.25, 0.50, 0.75], mode='full')

array([ 0.75,  2.  , -0.25, -0.25,  0.5 ,  1.5 , -0.5 , -1.25,  0.5 ])

### 2d

In [9]:
# --- reference: torch depthwise causal conv, exactly what ShortConvolution wraps ---
conv = nn.Conv1d(D, D, kernel_size=W, groups=D, padding=W - 1, bias=False)
conv.weight.data = theta.view(D, 1, W)

In [10]:
conv

Conv1d(4, 4, kernel_size=(4,), stride=(1,), padding=(3,), groups=4, bias=False)

In [12]:
Z_ref = conv(U.transpose(1, 2))[..., :T].transpose(1, 2)      # drop the W-1 non-causal tail
Z_ref

tensor([[[ 1.0000,  1.0000,  1.0000,  0.7500],
         [-0.5000,  1.5000,  2.5000,  2.0000],
         [ 1.0000,  3.0000, -0.2500, -0.2500],
         [ 2.7500,  1.7500,  3.7500, -0.2500],
         [ 0.2500,  0.7500,  1.2500,  0.5000],
         [ 3.5000,  0.5000,  3.7500,  1.5000]]], grad_fn=<TransposeBackward0>)

In [11]:
Y_ref = F.silu(Z_ref)
Y_ref

tensor([[[ 0.7311,  0.7311,  0.7311,  0.5094],
         [-0.1888,  1.2264,  2.3104,  1.7616],
         [ 0.7311,  2.8577, -0.1095, -0.1095],
         [ 2.5848,  1.4909,  3.6638, -0.1095],
         [ 0.1405,  0.5094,  0.9716,  0.3112],
         [ 3.3974,  0.3112,  3.6638,  1.2264]]], grad_fn=<SiluBackward0>)